# 2일차 실습 P6 — 짝 만들기 · 내 짝으로 학습

- **맨 위 준비 셀부터** 위에서 아래로 실행
- 셀 실행: 셀을 누르고 **Shift + Enter** 또는 셀 왼쪽 ▶
- Colab 에서 처음 열 때 경고 창이 뜨면 '계속' · 데이터는 내려받지 않음 · 준비 셀 · 문서 셀이 연습용 합성 문서를 만듦
- 빈칸은 `____` · 빈칸을 모두 바꾼 뒤 **그 셀부터 다시 실행**
- 학습 셀은 에폭마다 loss 한 줄씩 찍힘 · `학습 끝` 줄이 찍힐 때까지 기다림
- 막히면 `[안내]` 문장 → **에러 칸 맨 아래 줄** → 힌트 순서로 읽음
- 과제 셀에서 빨간 문법·실행 오류가 나면 확인 셀을 믿지 말 것 · 과제 셀을 고쳐 정상 실행한 뒤 확인 셀을 다시 실행할 것

## 에러를 읽는 법

- 에러 칸 첫 줄(`...Error    Traceback ...`)은 제목일 뿐 · 무엇이 틀렸는지는 **맨 아래 줄**
- 중간의 `---->` 화살표 줄 · numpy · torch 안쪽 칸은 처음엔 건너뜀 · **위에 찍힌 `[안내]` 문장과 맨 아래 줄부터** 읽음

| 맨 아래 줄에 보이는 말 | 뜻 | 먼저 볼 곳 |
|---|---|---|
| `name '____' is not defined` | 빈칸이 남음 | 화살표가 가리키는 줄 |
| `name 'make_doc'` · `'docs'` · `'use_ids'` · `'X'` is not defined | 그 이름을 만든 셀을 안 돌렸거나 위 셀이 에러로 멈춤 · 런타임이 끊김 | 준비 셀부터 순서대로 다시 |
| `name 'Clean'` is not defined | 내가 친 이름이 틀림 · 대소문자 · 이 셀에서 쓰는 이름은 모두 소문자 | 화살표가 가리키는 줄 |
| `all input arrays must have the same shape` | 잘라 낸 조각 크기가 서로 다름 · 문서 밖으로 나간 좌표가 있음 | 자르는 줄의 괄호 안 순서 `(판, 세로, 가로)` |

- **에러가 없는데** 확인 셀이 `[안내]` 를 찍으면 → 그 문장이 가리키는 줄부터
- 과제 셀에서 빨간 문법·실행 오류가 나면 확인 셀을 믿지 말 것 · 과제 셀을 고쳐 정상 실행한 뒤 확인 셀을 다시 실행할 것
- 학습 셀 맨 아래 줄이 `AssertionError: [안내] 확인 셀에서 …` → 확인 셀을 먼저 · P6-1 셀을 다시 실행했으면 확인 셀도 다시
- **에러가 없는데** 학습 뒤 `[안내]` 가 찍히고 그림의 my model 줄이 글자 없는 한 가지 밝기 네모 → 짝이 어긋났을 수 있음 · 확인 셀부터

## 준비

합성 문서 만들기 · 조각 자르기 · 모델 · 학습 함수 · 그림 함수 · 고치지 않고 실행

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

plt.rcParams.update({"figure.facecolor": "#1A222C", "axes.facecolor": "#1A222C", "savefig.facecolor": "#1A222C",   # 그림을 어두운 화면에 맞춤
                     "text.color": "#EEF2F6", "axes.labelcolor": "#EEF2F6", "xtick.color": "#EEF2F6", "ytick.color": "#EEF2F6", "axes.edgecolor": "#5C6A78"})

# ── 합성 문서 만들기 · 고치지 않고 실행 · 읽지 않아도 됨 ──
WORDS = ("the of and to in is was for on that with as by at from this be are or an report data model image paper result "
         "note page table value error clean noise sample layer filter record method letter number office meeting account "
         "review summary budget project quality section figure total").split()


def make_doc(seed, h=128, w=256, kinds=("gradient", "occlude", "saltpepper")):
    """문서 번호 하나 → (clean, dirty) · 값 0~1 · 같은 번호는 늘 같은 문서"""
    rng = np.random.default_rng(seed)
    size = int(rng.integers(12, 20))
    img = Image.new("L", (w, h), 255)
    draw = ImageDraw.Draw(img)
    font = ImageFont.load_default(size=size)
    y = int(rng.integers(2, max(3, size // 2)))
    while y + int(size * 1.45) <= h - 2:                      # 줄마다 단어를 이어 씀
        x, words = int(rng.integers(2, 12)), []
        while True:
            cand = " ".join(words + [WORDS[int(rng.integers(len(WORDS)))]])
            if x + draw.textlength(cand, font=font) > w - 4:
                break
            words = cand.split()
        if words:
            draw.text((x, y), " ".join(words), fill=0, font=font)
        y += int(size * 1.45)
    clean = np.asarray(img, dtype=np.float32) / 255.0
    d = clean.copy()
    if "gradient" in kinds:                                   # 한쪽으로 갈수록 어두워짐
        g = np.linspace(0, rng.uniform(0.1, 0.35), w, dtype=np.float32)[None, :]
        if rng.random() < 0.5:
            g = g[:, ::-1]
        d = d * (1 - g)
    if "occlude" in kinds:                                    # 회색 상자가 글자를 가림
        oh, ow = int(h * rng.uniform(0.1, 0.25)), int(w * rng.uniform(0.05, 0.15))
        oy, ox = int(rng.integers(0, h - oh)), int(rng.integers(0, w - ow))
        d[oy:oy + oh, ox:ox + ow] = rng.uniform(0.2, 0.6)
    if "saltpepper" in kinds:                                 # 검은 점 · 흰 점
        m = rng.random(d.shape)
        p = rng.uniform(0.002, 0.01)
        d[m < p / 2] = 0.0
        d[m > 1 - p / 2] = 1.0
    return clean, np.clip(d, 0, 1).astype(np.float32)


# ── 오늘 쓰는 크기 · 문서 번호 · 도구 ──
H, W, PS = 128, 256, 48                 # 문서 세로 · 가로 · 조각 한 변
train_ids = list(range(1000, 1024))     # 학습 문서 24장
eval_ids = list(range(2000, 2008))      # 평가 문서 8장 · 학습에 쓰지 않음


def crop(img, y, x):                    # (y, x) 를 왼쪽 위 모서리로 삼아 48×48 조각을 잘라 냄
    return img[y:y + PS, x:x + PS]


def to_batch(patches):                  # 조각 목록 → (조각 수, 1, 48, 48) 텐서
    return torch.tensor(np.stack(patches)).unsqueeze(1)


def make_model():                       # 1일차 Conv 오토인코더와 같은 구성 · 48 → 24 → 12 → 24 → 48
    return nn.Sequential(
        nn.Conv2d(1, 16, 3, 2, 1), nn.ReLU(),
        nn.Conv2d(16, 32, 3, 2, 1), nn.ReLU(),
        nn.ConvTranspose2d(32, 16, 3, 2, 1, output_padding=1), nn.ReLU(),
        nn.ConvTranspose2d(16, 1, 3, 2, 1, output_padding=1), nn.Sigmoid())


def train(X, Y, name, epochs=10):       # 입력 X · 정답 Y 로 학습 · 에폭마다 loss 한 줄
    torch.manual_seed(0)                # 모델마다 같은 출발점
    model = make_model()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    t = time.time()
    for epoch in range(1, epochs + 1):
        perm = torch.randperm(len(X))
        total = 0.0
        for k in range(0, len(X), 32):
            b = perm[k:k + 32]
            opt.zero_grad()
            loss = F.mse_loss(model(X[b]), Y[b])
            loss.backward()
            opt.step()
            total += loss.item() * len(b)
        print(f"{name}  에폭 {epoch:>2}  loss {total / len(X):.4f}")
    print(f"{name}  학습 끝 ({time.time() - t:.0f}초)")
    return model


def show_rows(rows, idx):               # rows = [(줄 이름, 조각 묶음), …] · 줄마다 같은 번호의 조각
    fig, axes = plt.subplots(len(rows), len(idx), figsize=(len(idx), 1.25 * len(rows)))
    for r, (name, T) in enumerate(rows):
        for c, j in enumerate(idx):
            axes[r, c].imshow(T[j, 0], cmap="gray", vmin=0, vmax=1)
            axes[r, c].axis("off")
        axes[r, 0].set_title(name, loc="left", fontsize=10)
    plt.tight_layout()
    plt.show()

## 가진 문서

학습 문서 · 평가 문서를 모두 만듦 · 고치지 않고 실행

In [ ]:
docs = {i: make_doc(i) for i in list(range(1000, 1024)) + list(range(2000, 2008))}   # 가진 문서 전부 · 문서 번호 → (clean, dirty)
print("가진 문서", len(docs), "장")

## P6-1. 짝 만들기

1. 위 **준비** 셀 → **가진 문서** 셀을 실행
2. 아래 셀의 빈칸 두 곳을 채움
   - (a) **정답 조각**을 자르는 줄 · 바로 윗줄의 입력 조각과 짝이 되게
   - (b) **학습에 쓸 문서 번호** 목록 · 가진 문서 32장 가운데 학습에 쓸 것만
3. 실행 → 학습 문서 수 · 입력 · 정답 모양이 찍힘 → 아래 **확인** 셀 실행 → `짝 확인 통과` 가 나오면 P6-2 로 · **P6-1 셀을 다시 실행했으면 확인 셀도 다시**
- 확인 셀이 `[안내]` 를 찍으면 그 문장부터 읽음 · 막히면 코드 셀 아래 **힌트 1** → **힌트 2** 순서로 하나씩 펼침
- 그래도 막히면 확인 셀 아래 **대체 셀**을 실행하고 P6-2 로

In [ ]:
# 문법 예시 (이 실습의 답 아님)
#   crop(img, 10, 20)                              → img 에서 세로 10 · 가로 20 자리를 왼쪽 위로 삼은 48×48 조각
#   [k for k in [1, 2, 3, 4] if k not in [2, 4]]   → [1, 3] · 뒤 목록에 든 것을 뺀 목록

X = Y = use_ids = None                     # 이 셀을 실행할 때마다 이전 결과를 지움 · 고치지 않음
짝_통과 = 대체_사용 = False                 # 확인 셀 통과 표시 · 대체 셀 사용 표시 · 고치지 않음


def make_pairs(ids, n=128, seed=2026):     # 문서 번호 목록 → (입력 조각 묶음, 정답 조각 묶음)
    rng = np.random.default_rng(seed)      # 좌표용 난수 · 고정
    X, Y = [], []
    for i in ids:
        clean, dirty = docs[i]
        ys, xs = rng.integers(0, H - PS + 1, n), rng.integers(0, W - PS + 1, n)   # 이 문서에서 자를 좌표 n개
        for y, x in zip(ys, xs):
            X.append(crop(dirty, y, x))    # 입력: dirty 조각
            Y.append(____)                 # (a) 정답 조각
    return to_batch(X), to_batch(Y)


use_ids = [i for i in docs if i not in ____]   # (b) 학습에 쓸 문서 번호
X, Y = make_pairs(use_ids)
print("학습 문서", len(use_ids), "장 · 입력", tuple(X.shape), " 정답", tuple(Y.shape))

<details><summary><b>힌트 1</b> — 막힐 때만 펼침</summary>

- (a) 바로 윗줄(입력 줄)과 견줌 · 같아야 하는 것: 자르는 **자리** · 달라야 하는 것: 자르는 **판**
- (b) 준비 셀의 '오늘 쓰는 크기 · 문서 번호 · 도구' 부분에 문서 번호 목록이 두 개 있음 · 학습에서 빼야 하는 쪽은 어느 것인가

</details>

<details><summary><b>힌트 2</b> — 힌트 1 로도 막힐 때</summary>

- (a) 입력 줄 `X.append(crop(dirty, y, x))` 에서 `X` → `Y` · `dirty` → 짝이 되는 깨끗한 판 이름 · 괄호 안 `y, x` 는 그대로
- (b) 빈칸에는 준비 셀의 평가 문서 번호 목록 이름 `eval_ids`

</details>

In [ ]:
# P6-1 확인 — 짝 · 문서 나눔 검사 · 고치지 않고 실행만 · 읽지 않아도 됨
# 약속(수업 설명과 같음): 학습 문서 1000~1023 · 평가 문서 2000~2007 · 문서마다 좌표 128개 · 좌표 난수 seed 2026
def 과제_문법_검사(*표시):                       # 과제 셀을 마지막으로 실행한 글자에 문법 오류가 있었는지 · 실행 기록(In)이 없으면 통과하지 않음
    기록 = globals().get("In")
    if not isinstance(기록, list) or len(기록) < 2:
        return False
    for 표 in 표시:
        for src in reversed(기록[:-1]):
            if 표 in src and "def 과제_문법_검사" not in src:
                try:
                    compile(src, "<과제 셀>", "exec")
                except SyntaxError:
                    return False
                break
    return True


def 짝_검사(use_ids, X, Y):
    if use_ids is None or X is None or Y is None:
        return "[안내] 위 P6-1 셀이 끝까지 실행되지 않았음 → 빈칸 ____ 두 곳을 모두 바꾸고 위 셀부터 다시 실행 · 준비 셀 · 가진 문서 셀을 안 돌렸으면 그것부터"
    ids = list(use_ids)
    if any(i in range(2000, 2008) for i in ids):
        return "[안내] 학습에 쓸 문서 번호에 평가 문서(2000 번대)가 들어 있음 → 평가 문서는 학습에 쓰지 않음 · 준비 셀의 문서 번호 목록 두 개를 볼 것"
    if sorted(ids) != list(range(1000, 1024)):
        return f"[안내] 학습 문서 번호가 {len(ids)}개 · 1000~1023 의 24장이 한 번씩 모두 들어가야 함"
    if not (torch.is_tensor(X) and torch.is_tensor(Y)):
        return "[안내] X · Y 가 텐서가 아님 → 위 셀 맨 끝 return 줄은 고치지 않음"
    if tuple(X.shape) != (3072, 1, 48, 48) or tuple(Y.shape) != (3072, 1, 48, 48):
        return f"[안내] 모양이 (3072, 1, 48, 48) 두 개가 아님 · 입력 {tuple(X.shape)} · 정답 {tuple(Y.shape)} → 문서 24장 × 좌표 128개"
    rng = np.random.default_rng(2026)
    RX, RY = [], []
    for i in ids:
        판 = make_doc(i)                   # 문서를 새로 만들어 비교 · docs 를 고쳐도 속지 않음
        ys, xs = rng.integers(0, H - PS + 1, 128), rng.integers(0, W - PS + 1, 128)
        for y, x in zip(ys, xs):
            RX.append(판[1][y:y + 48, x:x + 48])
            RY.append(판[0][y:y + 48, x:x + 48])
    RX, RY = torch.tensor(np.stack(RX)).unsqueeze(1), torch.tensor(np.stack(RY)).unsqueeze(1)
    X, Y = X.detach().float(), Y.detach().float()
    if not torch.equal(X, RX):
        return "[안내] 입력 조각 X 가 약속한 좌표의 dirty 조각과 다름 → 빈칸 두 곳 말고 다른 줄은 고치지 않음 · 반복 안에서 난수를 더 뽑으면 다음 문서부터 좌표가 밀림"
    if torch.equal(Y, X):
        return "[안내] 정답 Y 가 입력 X 와 똑같음 → 손상 조각을 정답으로 쓰면 모델이 배울 것은 '그대로 두기' · 정답은 어느 판에서 자르나"
    틀림 = int((Y != RY).flatten(1).any(1).sum())
    if 틀림:
        return f"[안내] 정답 조각 3072개 중 {틀림}개가 입력과 같은 자리의 clean 조각이 아님 → 입력 줄과 같은 좌표 · 괄호 안 순서 (판, 세로, 가로)"
    return "짝 확인 통과"


try:
    결과 = 짝_검사(use_ids, X, Y) if 과제_문법_검사('X = Y = use_ids = None') else "[안내] 과제 셀의 정상 실행을 확인할 수 없음(문법 오류 또는 실행 기록 없음) · 과제 셀을 고쳐 정상 실행한 뒤 확인 셀을 다시 실행 · 계속되면 런타임을 다시 시작하고 준비부터 실행"
except NameError:
    print("[안내] 위 P6-1 셀이 끝까지 실행되지 않았음 → 빈칸 ____ 두 곳을 모두 바꾸고 위 셀부터 다시 실행 · 준비 셀 · 가진 문서 셀을 안 돌렸으면 그것부터")
    raise
짝_통과 = 결과 == "짝 확인 통과"            # 통과를 찍을 때만 True · 학습 셀이 이 값을 봄
짝_출처 = "대체 셀" if 대체_사용 else "직접 만든 짝"
print(결과)

### 대체 셀 — 확인 셀이 `[안내]` 를 찍었고 막혔을 때만 실행

- P6-1 과 같은 X · Y · 학습 문서 목록을 다른 방법으로 만듦 · 실행하면 P6-2 로 넘어갈 수 있음
- `짝 확인 통과` 가 나왔으면 실행하지 않음 · 실행하면 내가 만든 X · Y 를 덮어쓰고 학습 셀 첫 줄에 `짝: 대체 셀` 이 찍힘

In [ ]:
# 대체 셀 — 막혔을 때만 · P6-1 과 같은 X · Y · use_ids 를 다른 방법으로 만듦
from numpy.lib.stride_tricks import sliding_window_view

use_ids = list(range(1000, 1024))                    # 학습 문서 24장
rng = np.random.default_rng(2026)
parts = []
for i in use_ids:
    pair = np.stack(docs[i])                         # (2, 128, 256) · 0번 = clean · 1번 = dirty
    ys, xs = rng.integers(0, H - PS + 1, 128), rng.integers(0, W - PS + 1, 128)
    parts.append(sliding_window_view(pair, (PS, PS), axis=(1, 2))[:, ys, xs])   # 두 판을 한꺼번에 · (2, 128, 48, 48)
both = torch.tensor(np.concatenate(parts, axis=1))   # (2, 3072, 48, 48)
X, Y = both[1].unsqueeze(1), both[0].unsqueeze(1)
print("대체 셀 · 학습 문서", len(use_ids), "장 · 입력", tuple(X.shape), " 정답", tuple(Y.shape))
짝_통과, 대체_사용, 짝_출처 = True, True, "대체 셀"   # 대체 셀로 만든 짝이라는 표시 · P6-1 셀을 다시 실행하면 지워짐

## P6-2. 내 짝으로 학습 · 결과 보기

아래 네 셀은 **고치지 않고** 차례로 실행

1. **짝 그림**: 위 줄 입력 · 아래 줄 정답 · 같은 칸끼리 글자 자리가 같은지 봄
2. **학습**: 수업과 같은 모델 · 10 에폭 · 에폭마다 loss 한 줄 · 확인 셀 통과(또는 대체 셀) 뒤에만 돎 · 첫 줄에 짝 출처
3. **평가 문서 조각 그림**: 학습에 쓰지 않은 평가 문서 8장 · 줄마다 dirty · clean · my model
4. **숫자**: `손상 그대로` · `내 모델` RMSE 두 줄(원본과의 차이 · 작을수록 원본에 가까움)
- 정상: 그림의 my model 줄에서 글자가 원래 자리에 보이고, 내 모델 RMSE 가 손상 그대로보다 작음
- 학습 셀이 `[안내]` 를 찍거나 my model 줄이 글자 없는 한 가지 밝기 네모 → P6-1 확인 셀부터 다시

In [ ]:
# 짝 그림 — 학습 문서 1000 의 앞 조각 8개 · 같은 칸이 한 짝
show_rows([("my input X", X), ("my answer Y", Y)], list(range(8)))

In [ ]:
# 학습 — 준비 셀의 train 함수 · 10 에폭 · 확인 셀 통과 뒤에만
assert globals().get("짝_통과"), "[안내] 확인 셀에서 '짝 확인 통과' 를 받은 뒤 학습 · P6-1 셀을 다시 실행했으면 확인 셀도 다시 · 막히면 대체 셀"
print("짝:", 짝_출처)
my_model = train(X, Y, "my pairs")

with torch.no_grad():
    end_mse = F.mse_loss(my_model(X), Y).item()      # 학습이 끝난 모델의 학습 조각 오차 · 경보용
if end_mse > 0.05:
    print("[안내] 학습이 끝났는데 학습 조각 오차가 높게 남음 → 짝이 어긋났을 수 있음 · P6-1 확인 셀부터 · 막히면 대체 셀")

In [ ]:
# 평가 문서 조각 — 학습에 쓰지 않은 문서 · 문서마다 조각 32개 · 좌표 난수 seed 7777(수업 셀과 같음)
test_ids = [i for i in docs if i not in use_ids]
assert len(test_ids) == 8, "[안내] 학습에 쓰지 않은 문서가 8장이 아님 → P6-1 확인 셀부터"
erng = np.random.default_rng(7777)
EX, EY = [], []
for i in test_ids:
    pair = np.stack(docs[i])                         # 0번 = clean · 1번 = dirty
    for _ in range(32):
        y, x = int(erng.integers(0, H - PS + 1)), int(erng.integers(0, W - PS + 1))
        EY.append(pair[0, y:y + PS, x:x + PS])
        EX.append(pair[1, y:y + PS, x:x + PS])
EX, EY = to_batch(EX), to_batch(EY)

with torch.no_grad():
    out = my_model(EX)

dark = (EY < 0.5).float().mean((1, 2, 3))
SHOW = [j * 32 + int(dark[j * 32:(j + 1) * 32].argmax()) for j in range(8)]   # 문서마다 글자가 가장 많은 조각 하나
show_rows([("dirty (input)", EX), ("clean (answer)", EY), ("my model", out)], SHOW)

In [ ]:
# 숫자 — 평가 조각 256개 · 조각마다 RMSE → 평균
def rmse(o):
    return ((o - EY) ** 2).mean((1, 2, 3)).sqrt().mean().item()


print(f"손상 그대로   RMSE {rmse(EX):.3f}")
print(f"내 모델       RMSE {rmse(out):.3f}")

## P6 마무리 — 세 줄 적기 (채점 아님)

이 칸을 두 번 눌러 `→` 뒤에 적음

1. 정답 조각을 입력과 같은 좌표에서 잘라야 하는 까닭 →
2. 평가 문서가 학습에 안 들어갔다는 것을 무엇으로 확인했나(복원 그림 말고) →
3. 그림에서 아직 안 된 복원 하나 →
4. 대체 셀을 썼나 · 썼다면 어디서 막혔나 →